In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

pd.set_option("display.max_columns", None)

plt.style.use("ggplot")

Load Dataset

In [2]:
DATA_PATH = "../datasets/raw"

train = pd.read_csv(DATA_PATH + "/train.csv")

train.head()

,sample_id,image_path,Sampling_Date,State,Species,Pre_GSHH_NDVI,Height_Ave_cm,target_name,target
0,ID1011485656__Dry_Clover_g,train/ID1011485656.jpg,2015/9/4,Tas,Ryegrass_Clover,0.62,4.6667,Dry_Clover_g,0.0000
1,ID1011485656__Dry_Dead_g,train/ID1011485656.jpg,2015/9/4,Tas,Ryegrass_Clover,0.62,4.6667,Dry_Dead_g,31.9984
2,ID1011485656__Dry_Green_g,train/ID1011485656.jpg,2015/9/4,Tas,Ryegrass_Clover,0.62,4.6667,Dry_Green_g,16.2751
3,ID1011485656__Dry_Total_g,train/ID1011485656.jpg,2015/9/4,Tas,Ryegrass_Clover,0.62,4.6667,Dry_Total_g,48.2735
4,ID1011485656__GDM_g,train/ID1011485656.jpg,2015/9/4,Tas,Ryegrass_Clover,0.62,4.6667,GDM_g,16.2750


Overview

In [3]:
print("Shape :", train.shape)

train.info()

train.describe(include="all")

Shape : (1785, 9)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1785 entries, 0 to 1784
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   sample_id      1785 non-null   object 
 1   image_path     1785 non-null   object 
 2   Sampling_Date  1785 non-null   object 
 3   State          1785 non-null   object 
 4   Species        1785 non-null   object 
 5   Pre_GSHH_NDVI  1785 non-null   float64
 6   Height_Ave_cm  1785 non-null   float64
 7   target_name    1785 non-null   object 
 8   target         1785 non-null   float64
dtypes: float64(3), object(6)
memory usage: 125.6+ KB


,sample_id,image_path,Sampling_Date,State,Species,Pre_GSHH_NDVI,Height_Ave_cm,target_name,target
count,1785,1785,1785,1785,1785,1785.000000,1785.000000,1785,1785.000000
unique,1785,357,28,4,15,NaN,NaN,5,NaN
top,ID1011485656__Dry_Clover_g,train/ID1011485656.jpg,2015/6/26,Tas,Ryegrass_Clover,NaN,NaN,Dry_Clover_g,NaN
freq,1,5,185,690,490,NaN,NaN,357,NaN
mean,NaN,NaN,NaN,NaN,NaN,0.657423,7.595985,NaN,24.782295
std,NaN,NaN,NaN,NaN,NaN,0.151972,10.273725,NaN,25.823738
min,NaN,NaN,NaN,NaN,NaN,0.160000,1.000000,NaN,0.000000
25%,NaN,NaN,NaN,NaN,NaN,0.560000,3.000000,NaN,4.818200
50%,NaN,NaN,NaN,NaN,NaN,0.690000,4.000000,NaN,18.200000
75%,NaN,NaN,NaN,NaN,NaN,0.770000,7.000000,NaN,35.940600


In [4]:
train.isnull().sum()

sample_id        0
image_path       0
Sampling_Date    0
State            0
Species          0
Pre_GSHH_NDVI    0
Height_Ave_cm    0
target_name      0
target           0
dtype: int64

In [5]:
missing = train.isnull().sum()

missing = missing[missing > 0]

missing

Series([], dtype: int64)

Unique tarfet Names

In [6]:
train["target_name"].value_counts()

target_name
Dry_Clover_g    357
Dry_Dead_g      357
Dry_Green_g     357
Dry_Total_g     357
GDM_g           357
Name: count, dtype: int64

Number Of Images

In [7]:
train["image_path"].nunique()

357

In [8]:
train["sample_id"].nunique()

1785

Convert Dataset to Wide Format

In [9]:
wide_df = train.pivot_table(
    index=[
        "image_path",
        "Sampling_Date",
        "State",
        "Species",
        "Pre_GSHH_NDVI",
        "Height_Ave_cm"
    ],
    columns="target_name",
    values="target"
).reset_index()

In [10]:
wide_df.head()

target_name,image_path,Sampling_Date,State,Species,Pre_GSHH_NDVI,Height_Ave_cm,Dry_Clover_g,Dry_Dead_g,Dry_Green_g,Dry_Total_g,GDM_g
0,train/ID1011485656.jpg,2015/9/4,Tas,Ryegrass_Clover,0.62,4.6667,0.0000,31.9984,16.2751,48.2735,16.2750
1,train/ID1012260530.jpg,2015/4/1,NSW,Lucerne,0.55,16.0000,0.0000,0.0000,7.6000,7.6000,7.6000
2,train/ID1025234388.jpg,2015/9/1,WA,SubcloverDalkeith,0.38,1.0000,6.0500,0.0000,0.0000,6.0500,6.0500
3,train/ID1028611175.jpg,2015/5/18,Tas,Ryegrass,0.66,5.0000,0.0000,30.9703,24.2376,55.2079,24.2376
4,train/ID1035947949.jpg,2015/9/11,Tas,Ryegrass,0.54,3.5000,0.4343,23.2239,10.5261,34.1844,10.9605


In [11]:
wide_df.shape

(357, 11)

In [12]:
wide_df.columns

Index(['image_path', 'Sampling_Date', 'State', 'Species', 'Pre_GSHH_NDVI',
       'Height_Ave_cm', 'Dry_Clover_g', 'Dry_Dead_g', 'Dry_Green_g',
       'Dry_Total_g', 'GDM_g'],
      dtype='object', name='target_name')

Saving in the procesed datasets

In [13]:
OUTPUT_PATH = "../datasets/processed"

Path(OUTPUT_PATH).mkdir(exist_ok=True)

wide_df.to_csv(
    OUTPUT_PATH + "/processed_train.csv",
    index=False
)

In [14]:
wide_df = wide_df.rename(columns={
    "Pre_GSHH_NDVI": "NDVI",
    "Height_Ave_cm": "Height_cm",
    "Sampling_Date": "Sampling_Date"
})

wide_df.columns.name = None